# Correlation Analysis of Tracks Data

This section performs a correlation analysis on the `cleaned_tracks.csv` dataset, which contains cleaned data from the previous analysis steps.

The focus will be on identifying relationships between numerical variables (audio features, lyrics statistics, track popularity).

In [1]:
import pandas as pd
import altair as alt
import numpy as np
import os

alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [2]:
dataset_path = os.path.join('..', 'dataset', 'cleaned_tracks.csv')
df = pd.read_csv(dataset_path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11158 entries, 0 to 11157
Data columns (total 45 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    11158 non-null  object 
 1   id_artist             11158 non-null  object 
 2   name_artist           11158 non-null  object 
 3   full_title            11158 non-null  object 
 4   title                 11158 non-null  object 
 5   featured_artists      3516 non-null   object 
 6   primary_artist        11158 non-null  object 
 7   language              10955 non-null  object 
 8   album                 9645 non-null   object 
 9   stats_pageviews       4637 non-null   float64
 10  swear_IT              11158 non-null  int64  
 11  swear_EN              11158 non-null  int64  
 12  swear_IT_words        11158 non-null  object 
 13  swear_EN_words        11158 non-null  object 
 14  year                  10720 non-null  float64
 15  month              

We filter the dataframe to include only relevant numerical columns for correlation analysis (grouped by type for clarity).

In [3]:
numerical_cols = [
    #metadata
    'popularity', 
    'stats_pageviews',
    'duration_ms',
    
    #audio features
    'bpm', 
    'centroid', 
    'rolloff', 
    'flux', 
    'rms', 
    'zcr', 
    'flatness', 
    'spectral_complexity', 
    'pitch', 
    'loudness',
    
    #lyrics stats
    'n_sentences', 
    'n_tokens', 
    'tokens_per_sent', 
    'char_per_tok', 
    'lexical_density', 
    'avg_token_per_clause',
    'swear_IT',
    'swear_EN'
]

cols_to_use = [col for col in numerical_cols if col in df.columns]

df_corr = df[cols_to_use].copy()

#ensure all data is numeric (coercing errors if any remain)
for col in df_corr.columns:
    df_corr[col] = pd.to_numeric(df_corr[col], errors='coerce')

print(f"selected {len(df_corr.columns)} numerical columns for analysis.")

selected 21 numerical columns for analysis.


We use the Pearson correlation coefficient to measure linear relationships. To visualize this in Altair (which requires long-format data), we will also transform the matrix and mask the upper triangle.

In [4]:
#calculate standard correlation matrix
corr_matrix = df_corr.corr()

#display the top 5 positive correlations (excluding self-correlation)
print("Top 5 Positive Correlations:")
c = corr_matrix.abs()
s = c.unstack()
so = s.sort_values(kind="quicksort", ascending=False)
print(so[so < 1.0].head(10)) #top 5 pairs (duplicates appear twice)

Top 5 Positive Correlations:
rms          loudness       0.995586
loudness     rms            0.995586
rolloff      zcr            0.969082
zcr          rolloff        0.969082
n_tokens     n_sentences    0.868337
n_sentences  n_tokens       0.868337
zcr          centroid       0.864166
centroid     zcr            0.864166
             rolloff        0.775364
rolloff      centroid       0.775364
dtype: float64


We now plot the correlation matrix using Altair.

In [5]:
#prepare data for altair
corr_long = corr_matrix.stack().reset_index()
corr_long.columns = ['Variable 1', 'Variable 2', 'Correlation']

corr_long['Correlation_Label'] = corr_long['Correlation'].apply(lambda x: f"{x:.2f}")

base = alt.Chart(corr_long).encode(
    x=alt.X('Variable 2', title=None, sort=cols_to_use),
    y=alt.Y('Variable 1', title=None, sort=cols_to_use)
)

heatmap = base.mark_rect().encode(
    color=alt.Color(
        'Correlation',
        scale=alt.Scale(scheme='redblue', domain=[-1, 1]),
        legend=alt.Legend(title="Correlation")
    ),
    tooltip=['Variable 1', 'Variable 2', alt.Tooltip('Correlation', format='.2f')]
)

text = base.mark_text(size=8).encode(
    text='Correlation_Label',
    color=alt.condition(
        alt.expr.abs(alt.datum.Correlation) > 0.5,  #if absolute correlation is high
        alt.value('white'),                    #use white text
        alt.value('black')                     #else use black text
    )
)

chart = (heatmap + text).properties(
    title='Pearson correlation matrix of numerical columns (audio features, lyrics stats, metadata)',
    width=800,
    height=800
)

chart.interactive()

alt.LayerChart(...)

using a 0.9 threshold, then we could delete one of the columns between rolloff and zcr, and one of the columns between loudness and rms:
- rolloff will be kept over zcr, as the rolloff is a specific frequency below which a certain percentage of the total spectral energy lies, while zcr is the rate at which the signal changes sign (Higher ZCR means noisier or more percussive sounds), so zcr is more sensitive to noise.
- loudness will be kept over rms, as the loudness is measured in decibels (dB), which is a logarithmic scale that matches how human ears perceive volume changes. rms is linear, so its distribution can be heavily skewed (loudness, being logarithmic, follows a more Gaussian distribution).